Trying to reimplement EnCodec by reading the paper https://arxiv.org/pdf/2210.13438

In [ ]:
import torch
import torch.nn as nn

conv = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=7, padding=3)

x = torch.randn(2, 1, 16000)  # batch of 2, mono audio, 1 second at 16kHz
out = conv(x)

print(out.shape) 

In [ ]:
conv_down = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=4, stride=2, padding=1)

x = torch.randn(2, 32, 16000)
out = conv_down(x)

print(out.shape)

In [ ]:
class ResidualUnit(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
                nn.Conv1d(
                    in_channels=channels, 
                    out_channels=channels, 
                    kernel_size=3, 
                    padding=1
                ),
                nn.Conv1d(
                    in_channels=channels, 
                    out_channels=channels, 
                    kernel_size=3, 
                    padding=1
                )
        )
        
    def forward(self, x):
        x  = x + self.conv(x)
        return x

res_unit = ResidualUnit(32)
x = torch.randn(2, 32, 16000)
out = res_unit(x)

out.shape

In [ ]:
class DownsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.residual_unit = ResidualUnit(in_channels)
        self.conv1d = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=stride*2,
            stride=stride, 
            padding=stride//2 if stride % 2 == 0 else stride//2 + 1,
        )
    
    def forward(self, x):
        x = self.residual_unit(x)
        x = self.conv1d(x)
        return x

downsample_block = DownsampleBlock(32, 32, 2)
x = torch.randn(2, 32, 16000)
out = downsample_block(x)

out.shape

In [ ]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=7, padding=3), # [B, 1, 24000] -> [B, 32, 24000] 
            DownsampleBlock(in_channels=32, out_channels=64, stride=2), # [B, 32, 24000] -> [B, 64, 12000] 
            DownsampleBlock(in_channels=64, out_channels=128, stride=4), # [B, 64, 12000] -> [B, 128, 3000] 
            DownsampleBlock(in_channels=128, out_channels=256, stride=5), # [B, 128, 3000] -> [B, 256, 600]  
            DownsampleBlock(in_channels=256, out_channels=512, stride=8), # [B, 256, 600] -> [B, 512, 75] 
        )
        self.lstm = nn.LSTM(input_size=512, hidden_size=512, num_layers=2, batch_first=True) # [B, 512, 75] -> [B, 75, 512]
        self.proj = nn.Conv1d(512, 128, kernel_size=1)

    def forward(self, x):
        x = self.cnn(x) # B, C, T
        x = x.permute(0, 2, 1)  # B, C, T -> B, T, C
        x, _ = self.lstm(x)
        x = x.permute(0, 2, 1)  # B, T, C -> B, C, T
        x = self.proj(x)
        return x

In [ ]:
encoder = Encoder().to("cuda")

In [ ]:
x = torch.randn(1, 1, 24000).to("cuda")
out = encoder(x)
out.shape

In [ ]:
upsample = nn.ConvTranspose1d(in_channels=64, out_channels=32, kernel_size=4, stride=2, padding=1)

x = torch.randn(1, 64, 8000)
out = upsample(x)
print(out.shape) 

In [ ]:
class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.conv1d_transposed = nn.ConvTranspose1d(
            in_channels=in_channels, 
            out_channels=out_channels, 
            kernel_size=stride*2, 
            stride=stride, 
            padding=stride // 2 if stride % 2 == 0 else stride // 2 + 1,
            output_padding=stride % 2
        )
        self.residual_unit = ResidualUnit(out_channels)
    
    def forward(self, x):
        x = self.conv1d_transposed(x)
        x = self.residual_unit(x)
        return x

x = torch.randn(1, 128, 75)
upsample = UpsampleBlock(128, 64, 5)
out = upsample(x)
print(out.shape)  

In [ ]:
x = torch.randn(1, 1, 100)  # 5 time steps
conv_t = nn.ConvTranspose1d(1, 1, kernel_size=10, stride=5, padding=3, output_padding=1)
out = conv_t(x)
print(out.shape)  # what do you expect?

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj_up = nn.ConvTranspose1d(
            in_channels=128, 
            out_channels=512, 
            kernel_size=1
        )
        self.lstm = nn.LSTM(input_size=512, hidden_size=512, num_layers=2, batch_first=True)
        self.layers = nn.Sequential(
            UpsampleBlock(in_channels=512, out_channels=256, stride=8), 
            UpsampleBlock(in_channels=256, out_channels=128, stride=5), 
            UpsampleBlock(in_channels=128, out_channels=64, stride=4),
            UpsampleBlock(in_channels=64, out_channels=32, stride=2), 
            nn.ConvTranspose1d(in_channels=32, out_channels=1, kernel_size=7, padding=3),
        )
    
    def forward(self, x):
        x = self.proj_up(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = x.permute(0, 2, 1)
        x = self.layers(x)
        return x

x = torch.randn(1, 128, 75)
decoder = Decoder()
out = decoder(x)
print(out.shape) 

In [ ]:
def find_nearest_cluster(embedding, centers):
    def get_distance2d(embedding, center):
        distance = torch.sqrt(torch.pow(embedding[0] - center[0], 2) + torch.pow(embedding[1] - center[1], 2))
        return distance

    nearest_center_idx = None
    best_distance = float("inf")
    for i, center in enumerate(centers):
        new_distance = get_distance2d(embedding, center)
        if new_distance < best_distance:
            nearest_center_idx = i
            best_distance = new_distance
    
    return nearest_center_idx

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, n_codes, dim):
        super().__init__()
        self.codebook = nn.Embedding(
            num_embeddings=n_codes, 
            embedding_dim=dim
        )
    
    def forward(self, z):
        # z shape: [B, T, dim]
        # 1. find nearest codebook entry for each vector
        # 2. quantize using straight-through estimator
        # 3. return quantized z, indices, and commitment loss
        # closest distance
        # (z - e)² = z² - 2ze + e²
        z_flatten = torch.flatten(z, start_dim=0, end_dim=1) # [B*T, dim]
        z_sum_squared = torch.sum(z_flatten**2, dim=1, keepdim=True)      # [B*T, 1]
        codebook_sum_squared = torch.sum(self.codebook.weight**2, dim=1)  # [n_codes]
        dot_product = z_flatten @ self.codebook.weight.T           # [B*T, n_codes]

        distances = z_sum_squared - 2 * dot_product + codebook_sum_squared  # [B*T, n_codes]
        indices = torch.argmin(distances, dim=1)  # [B*T]

        # look up the quantized vectors
        z_quantized = self.codebook(indices)  # [B*T, dim]
        z_quantized_st = z_flatten + (z_quantized - z_flatten).detach()
        commitment_loss = torch.mean((z_quantized.detach() - z_flatten) ** 2)
        z_quantized_st = z_quantized_st.reshape(z.shape)
        return z_quantized_st, indices, commitment_loss



z = torch.randn((4, 512, 128))
quantizier = VectorQuantizer(n_codes=4096, dim=128)
for value in quantizier(z):
    print(value.shape)


In [ ]:
z = torch.randn((4, 128, 256))
z.shape

In [ ]:
torch.flatten(z, start_dim=0, end_dim=1).shape

In [ ]:
z = torch.tensor([
    [1.1, 2.1],
    [2, 4],
])
print(z.shape)

z_sum = torch.sum(z**2, dim=1, keepdim=True)
z_sum.shape

In [ ]:
class RVQ(nn.Module):
    def __init__(self, n_levels, n_codes, dim):
        super().__init__()
        self.quantizers = nn.ModuleList(
            [VectorQuantizer(n_codes, dim) for _ in range(n_levels)]
        )
    
    def forward(self, z):
        residual = z
        total_loss = torch.zeros(1, device=z.device, dtype=torch.float32).squeeze()
        total_quantized = torch.zeros_like(z)
        all_indices = []
        for quantizer in self.quantizers:
            z_q, indices, loss = quantizer(residual)
            residual = residual - z_q
            total_loss += loss
            total_quantized += z_q
            all_indices.append(indices)
        
        return total_quantized, total_loss, all_indices



rvq = RVQ(n_levels=2, n_codes=8, dim=128)
z = torch.randn((4, 20, 128))

total_quantized, total_loss, all_indices = rvq(z)
print(total_quantized.shape)
print(total_loss.shape)
print(all_indices)

In [ ]:
class SimpleCodec(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.rvq = RVQ(n_levels=4, n_codes=4096, dim=128)
        self.decoder = Decoder()
    
    def forward(self, x):
        z = self.encoder(x)
        z = z.permute(0, 2, 1)
        z, commitment_loss, indices = self.rvq(z)
        z = z.permute(0, 2, 1)
        x_hat = self.decoder(z)
        return x_hat, commitment_loss

x = torch.randn(1, 1, 24000)
codec = SimpleCodec()
x_hat, commitment_loss = codec(x)
print(x_hat.shape) 
print(commitment_loss)

In [ ]:
import torchaudio

mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=24000, # how many samples in 1s audio
    n_fft=1024, # window size of fft
    hop_length=256, # hop length between windows
    n_mels=64 # frequency bins
)

x = torch.randn(1, 1, 24000)
print(mel(x).shape)

In [ ]:
def frequency_loss_fn(x, x_hat):
    total_loss_l1 = torch.tensor(0.0).to(x.device)
    total_loss_l2 = torch.tensor(0.0).to(x.device)
    frequency_bins = 64
    scales = [512, 1024, 2048, 4096]

    for window_len in scales:
        mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=24000, # how many samples in 1s audio
            n_fft=window_len, # window size of fft
            hop_length=window_len//4, # hop length between windows
            n_mels=frequency_bins # frequency bins
        ).to(x.device)
        mel = mel_transform(x)
        mel_hat = mel_transform(x_hat)
        mel_delta = mel - mel_hat
        total_loss_l1 += torch.mean(torch.abs(mel_delta))
        total_loss_l2 += torch.mean(mel_delta ** 2)
    
    return (total_loss_l1 + total_loss_l2) / len(scales)


frequency_loss_fn(x, x_hat)

In [ ]:
import torch
from datasets import load_dataset, Audio
from dotenv import load_dotenv

load_dotenv()

In [ ]:
dataset = load_dataset("shb777/gemini-flash-2.0-speech")
dataset = dataset["en"].shuffle()

dataset = dataset.remove_columns(["puck", "phoneme_length", "text"])

def filter_short(example):
    if example["kore"]["array"].shape[0] >= 24000:
        return True
    return False

dataset = dataset.filter(filter_short, num_proc=8)

In [ ]:
from torch.utils.data import Dataset
import numpy as np

class SpeechDataset(Dataset):
    def __init__(self, hf_dataset, clip_length=24000):
        self.data = hf_dataset
        self.clip_length = clip_length  # 1 second at 24kHz
    
    def __len__(self):
        return len(self.data)

    def random_crop(self, audio):
        if len(audio) < self.clip_length:
            return None
        if len(audio) == self.clip_length:
            return audio
        start = np.random.randint(0, len(audio) - self.clip_length)
        return audio[start:start+self.clip_length]
    
    def __getitem__(self, idx):
        audio_tensor = torch.tensor(self.data[idx]["kore"]["array"])
        audio_crop = self.random_crop(audio_tensor).unsqueeze(0)
        return audio_crop

In [ ]:
speech_dataset = SpeechDataset(dataset)
speech_dataset[0].shape

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(speech_dataset, batch_size=1, shuffle=True)


In [ ]:
import torchaudio

def save_sample(codec, dataset, path="sample.wav"):
    codec.eval()
    with torch.no_grad():
        x = dataset[0].unsqueeze(0).cuda()
        x_hat, _ = codec(x)
        torchaudio.save(path, x_hat.squeeze(0).cpu(), 24000)
    codec.train()

In [ ]:
import torch.optim as optim
from tqdm import tqdm

codec = SimpleCodec().cuda()
optimizer = optim.AdamW(codec.parameters(), lr=3e-4, betas=(0.5, 0.9), eps=1e-10)

def train_step(batch, codec, optimizer):
    # 1. get audio from batch
    x = batch.cuda()
    # 2. forward pass through codec
    x_hat, commitment_loss = codec(x)
    # 3. compute total loss (time + frequency + commitment)
    loss_time = torch.mean(torch.abs(x - x_hat))
    frequency_loss = frequency_loss_fn(x, x_hat)
    total_loss = loss_time + 0.000005 * frequency_loss + commitment_loss
    # 4. backprop
    total_loss.backward()
    # 5. optimizer step
    torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()
    # return loss value
    return total_loss.item()

# training loop
for epoch in range(3):
    pbar = tqdm(dataloader)
    for batch in pbar:
        loss = train_step(batch, codec, optimizer)
        pbar.set_postfix(loss=f"{loss:.4f}")
    save_sample(codec, speech_dataset)
    torch.save(codec.state_dict(), f"checkpoint_epoch_{epoch}.pt")

In [ ]:
from IPython.display import Audio, display

audio_array = speech_dataset[0]
display(Audio(audio_array, rate=24000))
audio_reconstructed, loss = codec(audio_array.cuda().unsqueeze(0))
display(Audio(audio_reconstructed.squeeze().cpu().detach().numpy(), rate=24000))

In [12]:
import torch
indices = torch.tensor([0, 1, 4, 5, 6, 7, 7])

n_codes = 8

n_codes_tensor = torch.arange(n_codes)

isin_tensor = torch.isin(n_codes_tensor, indices)



not_used_indices = (isin_tensor != True).nonzero(as_tuple=True)[0]
for index in not_used_indices:
    print(index)

torch.rand((128))

tensor(2)
tensor(3)


tensor([0.8945, 0.7753, 0.2549, 0.8484, 0.2150, 0.7171, 0.9569, 0.3806, 0.0266,
        0.0201, 0.7939, 0.7696, 0.4479, 0.9667, 0.4015, 0.0437, 0.5370, 0.8273,
        0.1097, 0.2959, 0.1908, 0.7909, 0.4506, 0.8981, 0.6876, 0.7001, 0.0240,
        0.5224, 0.0817, 0.7239, 0.5068, 0.3760, 0.1823, 0.6263, 0.9835, 0.9654,
        0.4524, 0.3674, 0.5765, 0.0617, 0.5467, 0.2571, 0.0640, 0.9149, 0.8154,
        0.0025, 0.7032, 0.7980, 0.5826, 0.5683, 0.1183, 0.3284, 0.3248, 0.6904,
        0.3563, 0.2092, 0.3338, 0.0450, 0.0871, 0.0220, 0.5513, 0.0961, 0.4762,
        0.2118, 0.0531, 0.2322, 0.8595, 0.7742, 0.9193, 0.2085, 0.9615, 0.0581,
        0.5455, 0.4661, 0.8443, 0.3737, 0.9379, 0.5486, 0.0075, 0.9203, 0.2398,
        0.4321, 0.4270, 0.8256, 0.5360, 0.2116, 0.4226, 0.6594, 0.8897, 0.2260,
        0.8203, 0.1682, 0.6877, 0.8552, 0.9534, 0.0791, 0.8126, 0.8389, 0.2550,
        0.2814, 0.1137, 0.1028, 0.1066, 0.6646, 0.1672, 0.8071, 0.3682, 0.8147,
        0.3062, 0.7469, 0.9575, 0.0597, 